# Recipe RAG Assistant — FREE Google Colab

## Student Notebook — No API Key, No Payment Required

We build the full RAG pipeline step by step, same as the syllabus project:

**Recipes → Chunks → Metadata → Embeddings → Vector Database → Similarity Search → Semantic Search → RAG → Answer → Citations → Evaluation**

No OpenAI/Anthropic API key is required — everything runs on Colab's free servers.

## How to use this notebook

For each stage:

**1. Understand the concept → 2. Run the code → 3. Observe the result → 4. Move to the next stage.**

Run cells top to bottom with Shift+Enter.

# 1. Install the required libraries

- `sentence-transformers` → free local embeddings
- `chromadb` → local vector database
- `transformers` → local LLM (runs on Colab, no API key)
- `accelerate` → model runtime

No paid API is needed.

In [ ]:
!pip install -q sentence-transformers chromadb transformers accelerate
print("✅ Installation complete")

# 2. Recipe Data

Instead of uploading PDFs, our knowledge base is a small set of recipes.
Each recipe becomes one "document," and we keep **metadata**:

- `name` = recipe name
- `ingredients` = ingredient list

This metadata will later be used for citations, exactly like `source` and `page` were used for the HR PDFs.

In [ ]:
recipes = [
    {"name": "Garlic Butter Pasta", "ingredients": ["pasta", "garlic", "butter", "parmesan", "parsley"], "instructions": "Boil pasta. Melt butter, saute minced garlic until fragrant. Toss pasta with garlic butter, top with parmesan and parsley."},
    {"name": "Veggie Fried Rice", "ingredients": ["rice", "egg", "carrot", "peas", "soy sauce", "spring onion"], "instructions": "Scramble egg in a hot pan, set aside. Stir-fry carrot and peas, add cold rice, soy sauce, and egg. Garnish with spring onion."},
    {"name": "Chickpea Curry", "ingredients": ["chickpeas", "tomato", "onion", "garlic", "curry powder", "coconut milk"], "instructions": "Saute onion and garlic, add curry powder, tomato, chickpeas, and coconut milk. Simmer 15 minutes."},
    {"name": "Egg Fried Toast", "ingredients": ["bread", "egg", "butter", "salt", "pepper"], "instructions": "Dip bread in beaten egg. Fry in buttered pan until golden on both sides. Season with salt and pepper."},
    {"name": "Chicken Stir Fry", "ingredients": ["chicken", "bell pepper", "broccoli", "soy sauce", "garlic", "ginger"], "instructions": "Stir-fry chicken until cooked. Add vegetables, garlic, ginger, and soy sauce. Cook until vegetables are tender-crisp."},
    {"name": "Lentil Soup", "ingredients": ["lentils", "carrot", "celery", "onion", "vegetable stock", "cumin"], "instructions": "Saute onion, carrot, celery. Add lentils, stock, and cumin. Simmer 25 minutes until lentils are soft."},
    {"name": "Caprese Salad", "ingredients": ["tomato", "mozzarella", "basil", "olive oil", "balsamic"], "instructions": "Slice tomato and mozzarella, layer with basil leaves. Drizzle with olive oil and balsamic glaze."},
    {"name": "Spinach Omelette", "ingredients": ["egg", "spinach", "cheese", "butter", "salt"], "instructions": "Beat eggs, wilt spinach in butter, pour eggs over spinach, add cheese, fold and cook until set."},
    {"name": "Beef Tacos", "ingredients": ["ground beef", "taco shells", "lettuce", "cheese", "salsa", "onion"], "instructions": "Cook beef with onion and taco seasoning. Fill shells with beef, lettuce, cheese, and salsa."},
    {"name": "Mushroom Risotto", "ingredients": ["rice", "mushroom", "onion", "vegetable stock", "parmesan", "butter"], "instructions": "Saute mushrooms and onion. Add rice, gradually stir in warm stock until creamy. Finish with parmesan and butter."},
    {"name": "Banana Pancakes", "ingredients": ["banana", "flour", "egg", "milk", "baking powder"], "instructions": "Mash banana, mix with egg, milk, flour, and baking powder. Cook spoonfuls on a griddle until bubbly, then flip."},
    {"name": "Greek Salad", "ingredients": ["cucumber", "tomato", "feta", "olives", "red onion", "olive oil"], "instructions": "Chop cucumber, tomato, and onion. Toss with olives, feta, and olive oil."},
    {"name": "Shrimp Scampi", "ingredients": ["shrimp", "garlic", "butter", "lemon", "pasta", "parsley"], "instructions": "Saute shrimp and garlic in butter. Toss with cooked pasta, lemon juice, and parsley."},
    {"name": "Black Bean Quesadilla", "ingredients": ["tortilla", "black beans", "cheese", "onion", "salsa"], "instructions": "Layer beans, cheese, and onion on tortilla, fold, and pan-fry until crispy and cheese melts."},
    {"name": "Tomato Basil Soup", "ingredients": ["tomato", "basil", "onion", "garlic", "vegetable stock", "cream"], "instructions": "Saute onion and garlic, add tomato and stock, simmer, blend smooth, stir in cream and basil."},
]

print("✅ Recipe data loaded")
print("Total recipes:", len(recipes))

# 3. Chunking

### Why chunk?

For long PDFs, we split text into smaller searchable pieces (chunks).

For this classroom lab, each **recipe is already short**, so we treat each whole recipe as **one chunk**. This mirrors the same `chunk_id` / `source` pattern used for PDFs — just simpler, since there's no need to split further.

In [ ]:
chunks = []
chunk_id = 0

for recipe in recipes:
    text = (
        f"{recipe['name']}. "
        f"Ingredients: {', '.join(recipe['ingredients'])}. "
        f"Instructions: {recipe['instructions']}"
    )

    chunks.append({
        "chunk_id": chunk_id,
        "text": text,
        "source": recipe["name"],
    })
    chunk_id += 1

print("✅ Chunking complete")
print("Total chunks:", len(chunks))

In [ ]:
for chunk in chunks[:3]:
    print("=" * 60)
    print("CHUNK:", chunk["chunk_id"])
    print("SOURCE:", chunk["source"])
    print("TEXT:", chunk["text"][:400])

# 4. Embeddings

An **embedding** converts text into a numerical vector representing its meaning.

We use the free local model:

`all-MiniLM-L6-v2`

So every recipe chunk can be compared with a user's craving without any API call.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("✅ Embeddings created")
print("Number of vectors:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

# 5. Vector Database

We need to store:

- chunks
- embeddings
- metadata

We use **ChromaDB**, a local vector database that runs entirely inside Colab.

In [ ]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="recipes"
)

collection.add(
    ids=[str(chunk["chunk_id"]) for chunk in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"source": chunk["source"]} for chunk in chunks]
)

print("✅ Vector database ready")
print("Stored chunks:", collection.count())

# 6. Ask a Natural-Language Question

Now we move to the query side.

Example:

> I have eggs and cheese, what can I make?

The question will also be converted into an embedding.

In [ ]:
question = input("What are you craving? ")

query_embedding = embedding_model.encode(
    [question]
)[0].tolist()

print("\nQUESTION:", question)
print("✅ Query embedding created")

# 7. Similarity Search + Semantic Search

**Similarity Search:** find vectors closest to the question vector.

**Semantic Search:** because embeddings represent meaning, relevant recipes can be found even when the exact words differ (e.g. "cheesy" can match a recipe with "parmesan").

We retrieve the Top-K most relevant chunks.

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=min(3, len(chunks))
)

print("✅ Similarity search complete")

In [ ]:
retrieved_chunks = []

for i, text in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i]

    item = {
        "text": text,
        "source": metadata["source"],
        "distance": distance
    }

    retrieved_chunks.append(item)

    print("=" * 60)
    print("RANK:", i + 1)
    print("DISTANCE:", round(distance, 4))
    print("SOURCE:", metadata["source"])
    print("TEXT:", text[:400])

# 8. Retrieval

This is the **Retrieval** part of RAG.

```text
User Craving
      ↓
Query Embedding
      ↓
Similarity Search
      ↓
Top-K Relevant Recipes
```

The system has found candidate recipes before generating an answer.

# 9. Augmentation

Now we put the retrieved recipes into a prompt.

This is the **Augmentation** part of RAG.

The language model will receive:

1. The user's craving/question
2. The retrieved recipe information

In [ ]:
context = "\n\n".join(
    f"RECIPE: {chunk['source']}\n"
    f"CONTENT: {chunk['text']}"
    for chunk in retrieved_chunks
)

prompt = (
    "You are a helpful cooking assistant.\n\n"
    "Answer the question using ONLY the recipes in the context below. "
    "Do not invent recipes that are not listed. If nothing fits well, say so.\n\n"
    "QUESTION:\n" + question + "\n\n"
    "RECIPE CONTEXT:\n" + context
)

print("✅ RAG prompt created")
print(prompt[:3500])

# 10. Generation — Local LLM

Now we perform **Generation**.

We use a small instruction-following model locally in Colab — completely free.

If available, use a GPU for speed:

**Runtime → Change runtime type → T4 GPU**

This keeps the whole project free of API charges.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading local LLM on:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("✅ Local LLM loaded")

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a friendly cooking assistant. Use only the supplied recipe context. Do not invent recipes."
    },
    {
        "role": "user",
        "content": prompt
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=4096
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

new_tokens = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("ANSWER")
print("=" * 60)
print(answer)

# 11. Citations

Because we kept `source` (recipe name) as metadata, we can show which recipes the answer came from — a **source-backed answer**, same idea as page citations for PDFs.

In [ ]:
print("RECIPES USED")
print("-" * 50)

seen = set()

for chunk in retrieved_chunks:
    if chunk["source"] not in seen:
        print(f"• {chunk['source']}")
        seen.add(chunk["source"])

# 12. Multi-Recipe Search

All recipes are stored in the same vector database.

Therefore one craving can search across the entire recipe collection at once, and only the most relevant ones are returned.

Try re-running from **Section 6** with a different craving and observe how the retrieved recipes change.

In [ ]:
print("Recipes in the knowledge base:")

for source in sorted(set(chunk["source"] for chunk in chunks)):
    print(" -", source)

print("\nTotal searchable chunks:", collection.count())

# 13. Evaluation

A RAG system must be evaluated.

### Retrieval
Did we retrieve the correct recipes?

### Answer
Did the answer correctly use those recipes?

### Groundedness
Are the answer's claims supported by the retrieved recipes?

### Citation correctness
Does the cited recipe actually support the answer?

In [ ]:
print("Retrieved evidence for evaluation:")
for rank, chunk in enumerate(retrieved_chunks, start=1):
    print(
        f"{rank}. {chunk['source']} | "
        f"distance={chunk['distance']:.4f}"
    )

# 14. Final RAG Pipeline

```text
Recipes
 ↓
Chunking
 ↓
Metadata
 ↓
Embeddings
 ↓
Vector Database
 ↓
User Craving
 ↓
Query Embedding
 ↓
Similarity / Semantic Search
 ↓
Relevant Recipes
 ↓
Augmented Prompt
 ↓
LLM Generation
 ↓
Source-backed Answer
 ↓
Citations
```

## Syllabus coverage

- ✅ Embeddings
- ✅ Vector Databases
- ✅ Semantic Search
- ✅ Chunking Strategies
- ✅ Metadata
- ✅ Similarity Search
- ✅ RAG Pipelines
- ✅ Evaluation

## Hands-on features

- ✅ Built-in recipe knowledge base
- ✅ Ask natural-language cravings
- ✅ Source-backed answers
- ✅ View citations
- ✅ Multi-recipe search

## Project idea

**Recipe assistant capable of answering "what can I cook" questions using a recipe knowledge base — fully free, no API key required.**